# Automated Poomsae Scoring System — Complete Project Workflow
# سامانهٔ خودکار امتیازدهی پومسه — مستند کامل فرایند پروژه

**English.** This notebook documents the complete workflow developed for the Poomsae scoring project, from raw multi-camera videos and judge score files to a validated, model-ready dataset and a fair comparison of multiple machine-learning and attention-based architectures. It explains what was built, why each decision was made, what files are produced, and what remains before deployment.

<div dir="rtl">
<b>فارسی.</b> این نوت‌بوک کل فرایندی را که برای پروژهٔ امتیازدهی پومسه توسعه داده شده مستند می‌کند؛ از ویدئوهای خام چنددوربینه و فایل‌های امتیاز داوران تا ساخت دیتاست آمادهٔ مدل‌سازی و مقایسهٔ منصفانهٔ چندین معماری یادگیری ماشین و مدل مبتنی بر توجه. در این سند توضیح داده می‌شود که چه بخش‌هایی ساخته شده‌اند، دلیل هر تصمیم چه بوده، چه فایل‌هایی تولید می‌شوند و برای رسیدن به مرحلهٔ استقرار چه کارهایی باقی مانده است.
</div>

> **Project status / وضعیت پروژه:** Video isolation, pose estimation, feature aggregation, label matching, and model-ready dataset preparation are complete. The model-comparison notebook is implemented; its final winner and numerical performance are determined only after running it successfully from beginning to end.
>
> <span dir="rtl">جداسازی ورزشکار، تخمین وضعیت بدن، استخراج و تجمیع ویژگی‌ها، اتصال برچسب‌ها و آماده‌سازی دیتاست تکمیل شده است. نوت‌بوک مقایسهٔ مدل‌ها نیز پیاده‌سازی شده؛ اما مدل برنده و نتایج عددی نهایی فقط پس از اجرای کامل آن مشخص می‌شوند.</span>

## 1. Problem definition and final objective

### English

The objective is to estimate a Poomsae athlete's judge score from video. Each performance is recorded from three camera angles—0°, 45°, and 135°—and has one judge-score file. The system is designed to predict two score components independently:

- **Accuracy score:** correctness and precision of techniques.
- **Presentation score:** speed and power, rhythm and tempo, and expression of energy.
- **Negative deductions:** subtracted from the component sum when present.
- **Total score:** `accuracy + presentation - negative deductions`.

The present dataset contains 54 labeled performances. All 54 negative-deduction values are zero, so the current model can learn accuracy and presentation but cannot learn when a deduction should occur. The modeled total is therefore the sum of predicted accuracy and predicted presentation.

### فارسی

<div dir="rtl">
هدف پروژه تخمین امتیاز داور برای اجرای پومسه از روی ویدئو است. هر اجرا از سه زاویهٔ ۰، ۴۵ و ۱۳۵ درجه ضبط شده و یک فایل امتیاز داوری دارد. سامانه به‌گونه‌ای طراحی شده است که دو جزء اصلی امتیاز را جداگانه پیش‌بینی کند:

- <b>امتیاز دقت (Accuracy):</b> درستی و دقت اجرای تکنیک‌ها.
- <b>امتیاز ارائه (Presentation):</b> سرعت و قدرت، ریتم و تمپو، و نمایش انرژی.
- <b>امتیاز منفی:</b> در صورت وجود، از مجموع دو بخش کم می‌شود.
- <b>امتیاز کل:</b> امتیاز دقت + امتیاز ارائه − امتیاز منفی.

دیتاست فعلی شامل ۵۴ اجرای برچسب‌خورده است. مقدار امتیاز منفی برای هر ۵۴ اجرا صفر است؛ بنابراین مدل فعلی می‌تواند دقت و ارائه را یاد بگیرد، اما هنوز قادر به یادگیری رخداد جریمه نیست. در نتیجه امتیاز کل پیش‌بینی‌شده فعلاً از جمع پیش‌بینی دقت و ارائه به‌دست می‌آید.
</div>

## 2. End-to-end architecture

```text
Raw videos (WPn-angle.MP4)          Judge logs (WPn.txt)
              │                              │
              ▼                              ▼
 Video discovery and validation       Score parsing and checks
              │                              │
              ▼                              │
 Athlete detection, tracking, isolation      │
              │                              │
              ▼                              │
 MediaPipe pose estimation                    │
              │                              │
              ▼                              │
 Frame-level biomechanical features           │
              │                              │
              ▼                              │
 View quality filtering and temporal summaries
              │                              │
              └──────────────┬───────────────┘
                             ▼
              One model-ready row per player
                             │
                             ▼
       Nested cross-validation model comparison
                             │
                             ▼
          Saved model bundle + inference function
```

<div dir="rtl">
<b>خلاصهٔ مسیر:</b> ویدئوی خام ابتدا شناسایی و اعتبارسنجی می‌شود؛ سپس ورزشکار اصلی تشخیص داده شده، در طول ویدئو ردیابی و از پس‌زمینه جدا می‌شود. روی ویدئوی برش‌خورده تخمین وضعیت بدن انجام می‌گیرد و ویژگی‌های بیومکانیکی هر فریم محاسبه می‌شوند. بعد از کنترل کیفیت، ویژگی‌های زمانی هر دوربین خلاصه شده و با امتیاز داور همان ورزشکار ادغام می‌شوند. در پایان، مدل‌های مختلف با اعتبارسنجی تودرتو مقایسه و مدل‌های نهایی ذخیره می‌شوند.
</div>

## 3. Input data organization

### English

The raw dataset follows the naming convention `WP<player number>-<camera angle>.MP4`, for example `WP12-0.MP4`, `WP12-45.MP4`, and `WP12-135.MP4`. The filename is important because it provides the join key between views and labels. Duplicate player-angle videos are rejected to prevent ambiguous matching.

Judge files follow the convention `WP<player number>.txt`. The parser extracts accuracy, presentation, negative score, and total score from the `Game Score` row. It verifies the scoring identity before accepting a label:

```text
total ≈ accuracy + presentation - negative
```

The dataset used in the prepared table contains:

| Item | Count |
|---|---:|
| Labeled player performances | 54 |
| Camera angles per performance | 3 |
| Total player-view videos | 162 |
| Reliable views after quality control | 152 |
| Excluded low-quality views | 10 |

### فارسی

<div dir="rtl">
نام ویدئوهای خام از الگوی <code>WP&lt;شماره ورزشکار&gt;-&lt;زاویه دوربین&gt;.MP4</code> پیروی می‌کند؛ برای مثال <code>WP12-0.MP4</code>، <code>WP12-45.MP4</code> و <code>WP12-135.MP4</code>. نام فایل کلید اتصال نماهای مختلف و برچسب امتیاز است. اگر برای یک ورزشکار و یک زاویه بیش از یک ویدئو پیدا شود، فرایند متوقف می‌شود تا اتصال مبهم ایجاد نشود.

فایل‌های داوری با الگوی <code>WP&lt;شماره ورزشکار&gt;.txt</code> نام‌گذاری شده‌اند. برنامه از سطر <code>Game Score</code> امتیاز دقت، ارائه، منفی و کل را استخراج می‌کند و پیش از پذیرش برچسب بررسی می‌کند که امتیاز کل تقریباً برابر «دقت + ارائه − منفی» باشد.

دیتاست نهایی آماده‌شده شامل ۵۴ اجرا، سه زاویه برای هر اجرا، در مجموع ۱۶۲ نمای ورزشکار، ۱۵۲ نمای قابل اعتماد و ۱۰ نمای حذف‌شده به علت کیفیت پایین است.
</div>

## 4. Initial exploration and prototype

### English

The notebook `front_athlete_preprocessing.ipynb` was used as the exploratory prototype. It established the basic feasibility of detecting the front athlete, suppressing background athletes, estimating pose, inspecting pose quality, parsing judge logs, and considering early baseline strategies. This exploratory work informed the reusable dataset pipelines that followed.

The prototype was useful for visual confirmation, but production-style dataset processing was moved into reusable Python modules and focused notebooks. This separation prevents one-off notebook state from becoming part of the data definition.

### فارسی

<div dir="rtl">
نوت‌بوک <code>front_athlete_preprocessing.ipynb</code> به‌عنوان نمونهٔ آزمایشی اولیه استفاده شد. در این مرحله امکان تشخیص ورزشکار جلویی، حذف ورزشکاران پس‌زمینه، تخمین وضعیت بدن، بررسی بصری کیفیت اسکلت، خواندن گزارش داوران و طراحی مدل‌های پایه بررسی شد.

این نمونه برای اثبات امکان‌پذیری و کنترل بصری مفید بود، اما پردازش نهایی کل دیتاست به ماژول‌های پایتون قابل استفادهٔ مجدد و نوت‌بوک‌های تخصصی منتقل شد. این جداسازی باعث می‌شود وضعیت موقت یک نوت‌بوک یا اجرای دستی به بخشی از تعریف دیتاست تبدیل نشود.
</div>

## 5. Athlete detection, tracking, and isolation

### English

The notebook `player_angle_isolation_pipeline.ipynb` and the reusable isolation pipeline process all discovered player-angle videos. Their responsibilities are:

1. Parse the player ID and camera angle from each filename.
2. Detect person instances with a YOLO segmentation model.
3. identify the athlete positioned on the competition mat rather than background people.
4. Track the same person throughout the video using detection continuity and ByteTrack-related tracking information.
5. Use lower-confidence recovery and short-gap interpolation when detection temporarily fails.
6. Apply the selected person's segmentation mask to remove the background.
7. Produce a padded, square, normalized crop for pose estimation.

Important default settings include a detection confidence of 0.35, recovery confidence of 0.15, output size of 640 pixels, recovery inference size of 960 pixels, maximum tracking gap of 8 frames, and crop padding of 25%. GPU inference can be selected with `cuda:0`; `auto` permits CPU fallback.

Each player-angle directory contains outputs such as:

```text
data/processed/players/WP12/angles/45/
├── isolated.mp4       # source-size frame with only the athlete visible
├── crop.mp4           # centered square input for pose estimation
├── tracking.csv       # frame-level boxes, IDs, confidence, and recovery state
└── metadata.json      # source and processing configuration
```

### فارسی

<div dir="rtl">
نوت‌بوک <code>player_angle_isolation_pipeline.ipynb</code> و ماژول جداسازی، تمام ویدئوهای ورزشکار-زاویه را پردازش می‌کنند. ابتدا شناسهٔ ورزشکار و زاویه از نام فایل استخراج می‌شود. سپس مدل قطعه‌بندی YOLO افراد حاضر در تصویر را تشخیص می‌دهد و ورزشکاری که روی زمین مسابقه قرار دارد از افراد پس‌زمینه تفکیک می‌شود. هویت همان فرد در طول ویدئو ردیابی می‌شود و در قطع‌های کوتاه تشخیص، از آستانهٔ بازیابی پایین‌تر و درون‌یابی فاصله‌های کوتاه استفاده می‌شود.

ماسک قطعه‌بندی برای سیاه‌کردن پس‌زمینه به‌کار می‌رود و یک برش مربعی همراه با حاشیه تولید می‌شود تا ورودی تخمین وضعیت بدن در تمام ویدئوها یکنواخت باشد. خروجی هر زاویه شامل ویدئوی جداشده، ویدئوی برش‌خورده، جدول ردیابی فریم‌به‌فریم و فایل متادیتا است.

تنظیمات مهم پیش‌فرض شامل اطمینان تشخیص ۰٫۳۵، اطمینان بازیابی ۰٫۱۵، اندازهٔ خروجی ۶۴۰ پیکسل، اندازهٔ بازیابی ۹۶۰ پیکسل، حداکثر فاصلهٔ ردیابی ۸ فریم و ۲۵ درصد حاشیهٔ برش است.
</div>

## 6. Pose estimation and frame-level representation

### English

The notebook `player_pose_estimation.ipynb` discovers each completed `crop.mp4` and runs MediaPipe Pose in video-tracking mode. The chosen configuration uses model complexity 2, detection confidence 0.5, tracking confidence 0.5, keypoint confidence 0.25, landmark smoothing, and a 640 × 640 working size.

MediaPipe returns 33 landmarks with normalized x, y, z, and visibility values. The pipeline also maps the relevant landmarks to a COCO-style 17-keypoint representation for reusable biomechanical calculations. For every frame it records whether a pose was detected and calculates features such as:

- left and right elbow angles;
- left and right knee angles;
- support-leg and kicking-leg knee angles;
- support side;
- torso lean;
- shoulder and hip tilt;
- stance width normalized by shoulder width;
- balance offset relative to the support ankle;
- wrist, ankle, and head offsets from the body centerline;
- mean landmark confidence;
- frame-to-frame motion energy.

Pose processing is resumable: completed views can be skipped instead of recomputed. Each pose directory contains:

```text
pose/
├── keypoints.csv      # frame-level landmarks
├── features.csv       # frame-level geometric and motion features
├── annotated.mp4      # skeleton overlay for quality review
└── metadata.json      # settings, FPS, frame counts, detection rate
```

### فارسی

<div dir="rtl">
نوت‌بوک <code>player_pose_estimation.ipynb</code> فایل‌های <code>crop.mp4</code> تکمیل‌شده را پیدا کرده و MediaPipe Pose را در حالت ردیابی ویدئویی اجرا می‌کند. تنظیمات انتخاب‌شده شامل پیچیدگی مدل ۲، اطمینان تشخیص و ردیابی ۰٫۵، آستانهٔ اطمینان نقاط کلیدی ۰٫۲۵، هموارسازی نقاط و اندازهٔ کاری ۶۴۰ در ۶۴۰ است.

MediaPipe برای هر فریم ۳۳ نقطهٔ بدن را همراه با مختصات نرمال‌شده و میزان دیده‌شدن تولید می‌کند. نقاط مرتبط برای محاسبات بیومکانیکی به نمایش ۱۷ نقطه‌ای مشابه COCO نیز نگاشت می‌شوند. زاویهٔ آرنج‌ها و زانوها، زانوی پای تکیه‌گاه و پای ضربه‌زن، سمت تکیه‌گاه، خم‌شدن تنه، شیب شانه و لگن، عرض استقرار، جابه‌جایی تعادل، فاصلهٔ اندام‌ها از خط مرکزی بدن، اطمینان نقاط و انرژی حرکت بین فریم‌ها محاسبه می‌شوند.

پردازش قابلیت ادامه‌دادن دارد و نماهایی که خروجی کامل دارند دوباره محاسبه نمی‌شوند. برای هر نما، فایل نقاط کلیدی، ویژگی‌های فریم‌به‌فریم، ویدئوی اسکلت‌گذاری‌شده و متادیتای کیفیت تولید می‌شود.
</div>

## 7. Pose quality control

### English

Raw pose predictions are not automatically trusted. In `prepare_model_dataset.ipynb`, every camera view is classified as reliable only when all of the following conditions hold:

| Quality condition | Required value |
|---|---:|
| Pose detection rate | at least 80% |
| Mean keypoint confidence for a usable frame | at least 0.50 |
| Usable-frame rate | at least 50% |
| Number of usable frames | at least 30 |

The results for the current data are:

| Angle | Reliable | Excluded |
|---:|---:|---:|
| 0° | 51 | 3 |
| 45° | 50 | 4 |
| 135° | 51 | 3 |
| **Total** | **152** | **10** |

An unreliable view is not replaced with zeros, because zero is a meaningful physical value for some measurements. Its pose summaries remain missing, while explicit availability, reliability, detection-rate, usable-rate, confidence, and duration columns are retained. Imputation is postponed until model training and fitted separately inside each training fold.

### فارسی

<div dir="rtl">
خروجی خام مدل وضعیت بدن بدون کنترل کیفیت وارد مدل امتیازدهی نمی‌شود. در نوت‌بوک آماده‌سازی دیتاست، یک نما فقط زمانی قابل اعتماد است که نرخ تشخیص حداقل ۸۰ درصد، اطمینان متوسط نقاط در فریم قابل استفاده حداقل ۰٫۵، نرخ فریم‌های قابل استفاده حداقل ۵۰ درصد و تعداد فریم‌های قابل استفاده حداقل ۳۰ باشد.

در دادهٔ فعلی از ۵۴ نمای زاویهٔ صفر، ۵۱ نما قابل اعتماد و ۳ نما حذف شده‌اند؛ برای زاویهٔ ۴۵ درجه ۵۰ نما قابل اعتماد و ۴ نما حذف شده‌اند؛ و برای زاویهٔ ۱۳۵ درجه ۵۱ نما قابل اعتماد و ۳ نما حذف شده‌اند. در مجموع ۱۵۲ نما قابل استفاده و ۱۰ نما کم‌کیفیت بوده‌اند.

ویژگی‌های یک نمای نامطمئن با صفر جایگزین نمی‌شوند، زیرا صفر برای بعضی اندازه‌گیری‌های فیزیکی مقدار واقعی و معنادار است. خلاصه‌های آن نما به‌صورت مقدار گمشده باقی می‌مانند، اما ستون‌های وضعیت دسترسی، قابلیت اعتماد، نرخ تشخیص، نرخ فریم قابل استفاده، اطمینان و مدت ویدئو حفظ می‌شوند. جایگذاری مقادیر گمشده فقط در زمان آموزش و درون هر fold انجام می‌شود تا نشت اطلاعات رخ ندهد.
</div>

## 8. Temporal aggregation: frames to one performance row

### English

A score is assigned to an entire performance, not to an individual frame. Therefore, frame-level features must be converted into one fixed-length player-level vector. For every reliable view and every numeric pose feature, the preparation pipeline calculates:

- median;
- standard deviation;
- 10th percentile;
- 90th percentile;
- median absolute velocity;
- 90th-percentile absolute velocity.

Velocity is calculated only between consecutive frames with positive elapsed time. Shoulder and hip tilt are treated as circular angles so crossing the −180°/180° boundary does not create a false large movement. The pipeline also records left-, right-, and unknown-support proportions and support-side switches per second.

The three camera views are kept separate. A 2D knee angle from 0° is not averaged with a knee angle from 45° because perspective changes its meaning. Feature names therefore carry a camera prefix such as `view_0_`, `view_45_`, or `view_135_`.

The final representation contains **112 columns per view × 3 views = 336 predictors per performance**.

### فارسی

<div dir="rtl">
امتیاز داور به کل اجرا تعلق دارد، نه به یک فریم منفرد. بنابراین ویژگی‌های فریم‌به‌فریم باید به یک بردار با طول ثابت برای هر ورزشکار تبدیل شوند. برای هر ویژگی عددی در هر نمای قابل اعتماد، میانه، انحراف معیار، صدک دهم، صدک نودم، میانهٔ سرعت مطلق و صدک نودم سرعت مطلق محاسبه می‌شود.

سرعت فقط بین فریم‌های متوالی و با اختلاف زمانی مثبت محاسبه می‌شود. زاویهٔ شانه و لگن به‌صورت دوری در نظر گرفته می‌شوند تا عبور از مرز منفی ۱۸۰ و مثبت ۱۸۰ درجه به‌اشتباه یک حرکت بسیار بزرگ ایجاد نکند. نسبت استفاده از پای چپ، راست یا نامشخص و تعداد تغییر پای تکیه‌گاه در هر ثانیه نیز ذخیره می‌شود.

نماهای سه دوربین جدا باقی می‌مانند. برای مثال زاویهٔ دوبعدی زانو در نمای صفر درجه با زاویهٔ همان زانو در نمای ۴۵ درجه میانگین‌گیری نمی‌شود، زیرا پرسپکتیو معنای هندسی آن را تغییر می‌دهد. در نتیجه نام هر ویژگی پیشوند زاویهٔ دوربین را دارد. نمایش نهایی برای هر اجرا شامل ۱۱۲ ویژگی در هر نما و در مجموع ۳۳۶ ویژگی است.
</div>

## 9. Label parsing, matching, and integrity checks

### English

The score parser reads `WP1.txt` through `WP54.txt`, normalizes player IDs, and extracts four target columns:

- `score_accuracy`
- `score_presentation`
- `score_negative`
- `score_total`

Before export, the workflow checks that player IDs are unique in labels and features, every labeled player has the required processed directories, the feature-label join is one-to-one, no player is lost, total scores lie within the expected range, and every recorded total follows the scoring formula within a small numerical tolerance.

The current target ranges are narrow: accuracy ranges from 2.76 to 3.20, presentation from 4.03 to 4.80, and total from 6.86 to 7.96. This narrow variation makes prediction harder: even modest point errors can correspond to weak R² values.

### فارسی

<div dir="rtl">
برنامه فایل‌های امتیاز <code>WP1.txt</code> تا <code>WP54.txt</code> را می‌خواند، شناسه‌ها را یکسان‌سازی می‌کند و چهار هدف دقت، ارائه، منفی و کل را استخراج می‌کند. قبل از خروجی‌گرفتن بررسی می‌شود که شناسه‌ها تکراری نباشند، هر ورزشکار برچسب‌خورده خروجی پردازش‌شده داشته باشد، اتصال ویژگی و برچسب یک‌به‌یک باشد، هیچ ورزشکاری حذف نشود، امتیاز کل در بازهٔ منطقی قرار گیرد و رابطهٔ محاسبهٔ امتیاز کل با تلورانس عددی کوچک برقرار باشد.

دامنهٔ برچسب‌ها محدود است: دقت از ۲٫۷۶ تا ۳٫۲۰، ارائه از ۴٫۰۳ تا ۴٫۸۰ و امتیاز کل از ۶٫۸۶ تا ۷٫۹۶ تغییر می‌کند. این تغییرات کم، مسئلهٔ پیش‌بینی را دشوارتر می‌کند؛ زیرا حتی خطای کوچک در واحد امتیاز می‌تواند به ضریب تعیین ضعیف یا منفی منجر شود.
</div>

## 10. Model-ready dataset outputs

### English

The notebook `prepare_model_dataset.ipynb` writes the following reproducible handoff:

```text
data/processed/model_ready/
├── prepared_dataset.csv  # player ID + targets + all predictors
├── features.csv          # player ID + 336 predictors; no score columns
├── targets.csv           # player ID + the four score columns
├── quality_report.csv    # one row per player-camera pair
└── metadata.json         # thresholds, dimensions, counts, and notes
```

The modeling unit is one performance. Frames are never randomly divided between train and test, because frames from one video are highly correlated and would create severe leakage. All validation is performed at player-performance level.

### فارسی

<div dir="rtl">
نوت‌بوک <code>prepare_model_dataset.ipynb</code> پنج خروجی قابل بازتولید می‌سازد: دیتاست کامل شامل شناسه، اهداف و ویژگی‌ها؛ فایل ویژگی‌ها بدون ستون امتیاز؛ فایل اهداف؛ گزارش کیفیت هر زوج ورزشکار-دوربین؛ و فایل متادیتا شامل آستانه‌ها، ابعاد و توضیحات.

واحد مدل‌سازی یک اجرای کامل است. فریم‌های یک ویدئو هرگز به‌طور تصادفی بین آموزش و آزمون تقسیم نمی‌شوند، زیرا فریم‌های یک اجرا بسیار هم‌بسته‌اند و چنین تقسیمی نشت اطلاعات شدیدی ایجاد می‌کند. تمام اعتبارسنجی‌ها در سطح اجرای ورزشکار انجام می‌شوند.
</div>

## 11. Why the modeling strategy is conservative

### English

The modeling problem has a difficult ratio: 54 rows versus 336 input columns. A flexible model can memorize training players very easily. For this reason, `build_score_model.ipynb` does not rely on a single train/test split and does not report training accuracy.

The preprocessing required by a model is contained inside its pipeline:

1. median imputation for missing views or measurements;
2. optional missingness indicators;
3. removal of constant columns;
4. standardization when required by the architecture;
5. supervised feature selection for high-dimensional linear, kernel, and MLP models;
6. fitting the regressor.

Each of these learned operations is fitted using training data only. Performing imputation, scaling, or feature selection once on the full dataset before validation would leak information from the test folds.

### فارسی

<div dir="rtl">
نسبت ابعاد این مسئله دشوار است: فقط ۵۴ سطر در برابر ۳۳۶ ورودی. یک مدل انعطاف‌پذیر می‌تواند ورزشکاران آموزشی را به‌سادگی حفظ کند. به همین دلیل نوت‌بوک مدل‌سازی به یک تقسیم آموزش/آزمون اکتفا نمی‌کند و عملکرد روی دادهٔ آموزش را به‌عنوان معیار واقعی گزارش نمی‌دهد.

پیش‌پردازش هر مدل داخل Pipeline آن انجام می‌شود: جایگذاری میانه برای مقادیر گمشده، افزودن نشانگر گمشدگی در صورت نیاز، حذف ستون‌های ثابت، استانداردسازی، انتخاب ویژگی برای مدل‌های پُربعد و در پایان برازش رگرسور. تمام این عملیات فقط روی دادهٔ آموزش هر fold یاد گرفته می‌شوند. اجرای این مراحل یک‌بار روی کل دیتاست، پیش از اعتبارسنجی، باعث نشت اطلاعات از بخش آزمون می‌شود.
</div>

## 12. Architectures included in the comparison

### English

| Architecture | Purpose and suitability |
|---|---|
| Mean baseline | Essential reference; predicts the training-fold mean. A learned model must improve on it. |
| Ridge regression | Strong small-data baseline; shrinks correlated coefficients and works well with many features. |
| Elastic Net | Combines L1 and L2 regularization and can produce a sparser model. |
| RBF Support Vector Regression | Tests smooth nonlinear relationships while retaining strong regularization. |
| Random Forest | Tests nonlinear rules and feature interactions using bagged decision trees. |
| Extra Trees | Adds stronger tree randomization, which can reduce variance relative to a single tree. |
| Histogram Gradient Boosting | Builds a sequence of shallow trees to correct previous residuals. |
| MLP | A small feed-forward neural network; included to test learned nonlinear combinations of selected features. |
| Camera-view attention Transformer | Treats the three cameras as three tokens and learns cross-view interactions and attention weights. |

The attention architecture is intentionally small. Each camera contributes an equal 112-feature token. A shared linear projection maps tokens into a compact embedding, learned camera embeddings identify 0°, 45°, and 135°, one Transformer encoder layer exchanges information, attention pooling combines the views, and a regression head produces one score component.

This is a **view-level Transformer**, not a temporal video Transformer. A temporal Transformer would require aligned frame- or movement-level sequences and many more labeled performances.

### فارسی

<div dir="rtl">
برای مقایسه، یک مدل میانگین به‌عنوان خط پایه، Ridge، Elastic Net، رگرسیون بردار پشتیبان با هستهٔ RBF، جنگل تصادفی، Extra Trees، گرادیان بوستینگ هیستوگرامی، شبکهٔ عصبی MLP و یک ترنسفورمر توجه بین نماها در نظر گرفته شده‌اند.

Ridge و Elastic Net برای دادهٔ کم و ویژگی‌های هم‌بسته مناسب‌اند. SVR روابط غیرخطی هموار را بررسی می‌کند. مدل‌های درختی تعامل‌ها و قواعد غیرخطی را بدون نیاز به مقیاس‌بندی می‌آزمایند. MLP ترکیب‌های غیرخطی یادگرفتنی را بررسی می‌کند، اما با ۵۴ نمونه خطر بیش‌برازش دارد.

در مدل توجه، هر دوربین یک token شامل ۱۱۲ ویژگی است. یک نگاشت مشترک هر token را به فضای کوچک‌تری می‌برد؛ embeddingهای قابل یادگیری زاویهٔ دوربین را مشخص می‌کنند؛ یک لایهٔ Transformer اطلاعات نماها را ترکیب می‌کند؛ attention pooling اهمیت نسبی نماها را یاد می‌گیرد؛ و سر رگرسیون یکی از اجزای امتیاز را تولید می‌کند. این مدل ترنسفورمر بین نماهاست، نه ترنسفورمر زمانی روی فریم‌های ویدئو.
</div>

## 13. Nested cross-validation and fair model selection

### English

All architectures are compared using the same repeated outer folds. The procedure is:

1. Hold out one outer fold as unseen test players.
2. On the remaining outer-training players, run an inner K-fold search over a deliberately small hyperparameter grid.
3. Refit the best inner configuration on the complete outer-training fold.
4. Predict the untouched outer-test players.
5. Repeat for all folds and repetitions.
6. Average repeated out-of-fold predictions for each player.
7. Rank architectures using these player-level predictions.

The same outer indices are reused for every architecture, making differences more comparable. Hyperparameter grids are kept small because an enormous search on 54 cases increases selection noise rather than creating new information.

The notebook supports two execution modes:

- `QUICK_RUN = True`: one value from each grid and fewer repetitions, used only to verify the workflow.
- `QUICK_RUN = False`: the full comparison used for final analysis.

### فارسی

<div dir="rtl">
تمام معماری‌ها با foldهای بیرونی یکسان مقایسه می‌شوند. در هر مرحله، یک fold بیرونی به‌عنوان دادهٔ کاملاً دیده‌نشده کنار گذاشته می‌شود. روی بخش آموزشی باقی‌مانده، اعتبارسنجی داخلی بهترین ابرپارامتر را انتخاب می‌کند. سپس مدل منتخب روی کل بخش آموزشی بیرونی دوباره برازش شده و برای fold آزمون پیش‌بینی می‌کند. این فرایند برای همهٔ foldها و تکرارها انجام شده و پیش‌بینی‌های خارج از fold هر ورزشکار میانگین‌گیری می‌شوند.

استفاده از تقسیم‌های یکسان برای همهٔ مدل‌ها مقایسه را منصفانه‌تر می‌کند. فضای جست‌وجو عمداً کوچک است؛ زیرا جست‌وجوی بسیار بزرگ روی ۵۴ نمونه بیشتر باعث انتخاب تصادفی و ناپایدار می‌شود تا بهبود واقعی.

حالت <code>QUICK_RUN = True</code> فقط برای بررسی صحت اجرای کد است. برای تحلیل نهایی باید مقدار آن <code>False</code> باشد تا جست‌وجوی کامل‌تر و تکرارهای اعتبارسنجی اجرا شوند.
</div>

## 14. Evaluation metrics and interpretation

### English

Each architecture is evaluated for accuracy, presentation, and derived total score using:

- **MAE (Mean Absolute Error):** primary metric, measured directly in score points.
- **RMSE (Root Mean Squared Error):** gives more weight to occasional large errors.
- **R²:** measures improvement over a constant mean prediction; it may be negative on small, narrow-range data.

A model should first beat the mean baseline. The notebook ranks models using mean MAE rank across accuracy, presentation, and total. If two models are close, the simpler and more stable model—usually Ridge—should be preferred unless repeated validation provides clear evidence for the more complex model.

The Transformer must not be selected merely because attention is modern. Model choice is determined by held-out performance, stability across folds, operational complexity, and interpretability.

### فارسی

<div dir="rtl">
هر معماری برای امتیاز دقت، ارائه و امتیاز کل مشتق‌شده با سه معیار ارزیابی می‌شود. MAE معیار اصلی است و خطا را مستقیماً در واحد امتیاز نشان می‌دهد. RMSE به خطاهای بزرگ وزن بیشتری می‌دهد. ضریب R² میزان بهبود نسبت به پیش‌بینی ثابت میانگین را اندازه می‌گیرد و در دادهٔ کوچک با دامنهٔ محدود ممکن است منفی شود.

اولین شرط مفیدبودن یک مدل، بهترشدن نسبت به خط پایهٔ میانگین است. رتبه‌بندی با میانگین رتبهٔ MAE در سه هدف انجام می‌شود. اگر دو مدل عملکرد نزدیک داشته باشند، مدل ساده‌تر و پایدارتر—معمولاً Ridge—انتخاب مناسب‌تری است، مگر آنکه اعتبارسنجی تکرارشده برتری واضح مدل پیچیده را نشان دهد.

ترنسفورمر نباید صرفاً به‌دلیل جدیدبودن مفهوم attention انتخاب شود. انتخاب باید بر اساس عملکرد دادهٔ دیده‌نشده، پایداری بین foldها، هزینهٔ عملیاتی و قابلیت تفسیر باشد.
</div>

## 15. Final fitting, saved artifacts, and inference

### English

After nested evaluation, every architecture is tuned and refitted on all 54 labeled performances. These final fits are deployment candidates; their training predictions are not performance estimates. The notebook saves:

```text
data/processed/models/score_model_comparison/
├── score_model_comparison.joblib  # all fitted models and metadata
├── out_of_fold_predictions.csv    # unbiased per-player component predictions
├── model_metrics.csv              # MAE, RMSE, R², and ranks
├── model_ranking.csv              # architecture-level ranking
├── nested_tuning_history.csv      # best parameters in each outer fold
├── final_search_results.csv       # full-data refit parameters
└── comparison_summary.json        # compact experiment description
```

The serialized bundle stores the exact feature order, camera angles, feature suffixes, target bounds, all fitted pipelines, the recommended model name, total-score formula, and list of training players. The inference helper validates required columns, reorders them correctly, predicts both components, clips them to valid ranges, and calculates total score.

A new video cannot be passed directly to the score model. It must first go through the same isolation, pose estimation, quality filtering, and aggregation pipeline so its feature schema matches the training data exactly.

### فارسی

<div dir="rtl">
پس از ارزیابی تودرتو، هر معماری با تمام ۵۴ اجرای برچسب‌خورده تنظیم و برازش می‌شود. این مدل‌های نهایی گزینه‌های استقرار هستند، اما پیش‌بینی آن‌ها روی دادهٔ آموزش معیار عملکرد واقعی محسوب نمی‌شود.

خروجی‌ها شامل بستهٔ همهٔ مدل‌ها، پیش‌بینی‌های خارج از fold، جدول معیارها، رتبه‌بندی مدل‌ها، تاریخچهٔ تنظیم ابرپارامترها، نتایج برازش نهایی و خلاصهٔ آزمایش است. بستهٔ ذخیره‌شده ترتیب دقیق ویژگی‌ها، زاویه‌ها، محدودهٔ اهداف، Pipelineهای برازش‌شده، نام مدل پیشنهادی، فرمول امتیاز کل و فهرست ورزشکاران آموزشی را نگه می‌دارد.

یک ویدئوی جدید را نمی‌توان مستقیماً به مدل امتیاز داد. ویدئو باید دقیقاً همان مراحل جداسازی ورزشکار، تخمین وضعیت بدن، کنترل کیفیت و تجمیع زمانی را طی کند تا ساختار ویژگی آن با دادهٔ آموزش یکسان باشد.
</div>

## 16. Validation, testing, and reproducibility safeguards

### English

Several safeguards were added throughout the project:

- configuration dataclasses reject invalid dimensions, confidence thresholds, codecs, or durations;
- duplicate videos and labels are detected;
- player-label matching is explicitly one-to-one;
- score totals are checked against their components;
- completed isolation and pose outputs can be discovered and resumed;
- metadata records processing settings and quality counts;
- low-quality views remain identifiable rather than silently becoming zeros;
- cross-validation preprocessing is fold-local;
- fixed random seeds improve repeatability;
- baseline predictions are retained in the final comparison;
- reusable modules are covered by unit tests under `tests/`.

The environment dependencies are declared in `pyproject.toml` and `requirements.txt`. Core packages include NumPy, pandas, OpenCV, Ultralytics, MediaPipe, scikit-learn, PyTorch, Matplotlib, and joblib.

### فارسی

<div dir="rtl">
در طول پروژه چندین کنترل برای قابلیت اعتماد و بازتولیدپذیری اضافه شده است: تنظیمات نامعتبر رد می‌شوند؛ ویدئوها و برچسب‌های تکراری شناسایی می‌شوند؛ اتصال ورزشکار و برچسب به‌صورت صریح یک‌به‌یک است؛ فرمول امتیاز کل بررسی می‌شود؛ پردازش‌های تکمیل‌شده قابل شناسایی و ادامه‌دادن هستند؛ تنظیمات و شاخص‌های کیفیت در متادیتا ثبت می‌شوند؛ نماهای کم‌کیفیت قابل تشخیص باقی می‌مانند؛ پیش‌پردازش اعتبارسنجی فقط درون fold آموزش انجام می‌شود؛ seedهای ثابت به تکرارپذیری کمک می‌کنند؛ خط پایه در نتایج نگه داشته می‌شود؛ و ماژول‌های اصلی دارای آزمون واحد در پوشهٔ <code>tests</code> هستند.

وابستگی‌های محیط در فایل‌های <code>pyproject.toml</code> و <code>requirements.txt</code> تعریف شده‌اند و شامل کتابخانه‌های اصلی پردازش تصویر، تخمین وضعیت بدن، یادگیری ماشین، شبکهٔ عصبی، رسم نمودار و ذخیرهٔ مدل هستند.
</div>

## 17. Current limitations

### English

1. **Small sample size:** 54 labeled rows are insufficient for reliable high-capacity deep learning.
2. **High dimensionality:** 336 predictors create a strong overfitting risk.
3. **Narrow score distribution:** most performances receive similar scores, limiting learnable variance.
4. **No deduction examples:** the model cannot learn penalties while every negative target equals zero.
5. **Player-level summaries lose sequence order:** medians and percentiles preserve distribution and motion magnitude but not the exact order of techniques.
6. **2D perspective effects:** keeping views separate reduces the problem but does not reconstruct true 3D motion.
7. **Potential judge noise:** one final score may contain subjectivity and does not identify which movement caused an error.
8. **No independent external test set:** nested cross-validation is appropriate for development, but final claims require data from new athletes, events, cameras, and judges.
9. **Attention interpretability is limited:** a high view-attention weight does not prove that a camera or biomechanical feature caused the judge score.

### فارسی

<div dir="rtl">
محدودیت اصلی تعداد کم نمونه‌هاست؛ ۵۴ سطر برای یادگیری عمیق پُرظرفیت کافی نیست. تعداد ۳۳۶ ویژگی خطر بیش‌برازش را افزایش می‌دهد و دامنهٔ محدود امتیازها نیز تنوع قابل یادگیری را کم می‌کند. چون هیچ نمونهٔ جریمه‌ای وجود ندارد، مدل امتیاز منفی قابل آموزش نیست.

خلاصه‌سازی در سطح اجرا ترتیب دقیق تکنیک‌ها را از بین می‌برد؛ نماهای دوبعدی نیز حرکت سه‌بعدی واقعی را به‌طور کامل بازسازی نمی‌کنند. امتیاز نهایی داور می‌تواند دارای نویز و ذهنیت باشد و مشخص نمی‌کند خطا در کدام حرکت رخ داده است. همچنین هنوز مجموعه‌آزمون خارجی مستقلی از مسابقه، دوربین، ورزشکار و داور جدید وجود ندارد. وزن attention نیز رابطهٔ علّی را اثبات نمی‌کند.
</div>

## 18. Recommended next steps

### English

The next development priorities should be:

1. Run the model-comparison notebook first in quick mode, then in full mode, and inspect failures and fold stability.
2. Confirm that the best learned architecture consistently beats the mean baseline for both components and total score.
3. Collect substantially more performances covering beginners through elite athletes and the full score range.
4. Include multiple judges or judge-level scores to estimate label agreement and uncertainty.
5. Collect non-zero deductions and record their type and time interval.
6. Segment performances into named Poomsae movements and create movement-level targets or error labels.
7. Add temporal features such as phase duration, synchronization, acceleration, pauses, and trajectory consistency.
8. Investigate multi-view 3D pose reconstruction or calibrated geometry.
9. Only after expanding the data, compare temporal convolutional networks, recurrent networks, and frame/movement-level Transformers.
10. Reserve a genuinely external test set and report confidence intervals, subgroup performance, and calibration before deployment.

### فارسی

<div dir="rtl">
گام بعدی این است که نوت‌بوک مقایسهٔ مدل‌ها ابتدا در حالت سریع و سپس کامل اجرا شود و پایداری نتایج در foldهای مختلف بررسی گردد. مدل یادگرفته‌شده باید برای هر دو جزء و امتیاز کل به‌طور پایدار از خط پایه بهتر باشد.

مهم‌ترین پیشرفت آینده افزایش جدی تعداد اجراها و پوشش کل دامنهٔ مهارتی و امتیازی است. بهتر است امتیاز چند داور به‌صورت جداگانه ذخیره شود تا توافق و عدم‌قطعیت برچسب‌ها اندازه‌گیری گردد. نمونه‌های جریمه همراه با نوع و بازهٔ زمانی آن‌ها نیز باید جمع‌آوری شوند.

تقسیم اجرا به حرکات نام‌گذاری‌شدهٔ پومسه، ساخت برچسب خطا در سطح حرکت، افزودن ویژگی‌های مدت فاز، هم‌زمانی، شتاب، مکث و ثبات مسیر، و بررسی بازسازی وضعیت سه‌بعدی چنددوربینه ارزش بالایی دارند. مقایسهٔ شبکه‌های زمانی و ترنسفورمر روی توالی فریم یا حرکت باید پس از افزایش داده انجام شود. پیش از استقرار نیز یک مجموعه‌آزمون خارجی واقعی و گزارش فاصلهٔ اطمینان، عملکرد زیرگروه‌ها و کالیبراسیون ضروری است.
</div>

## 19. Notebook and module map

| File | Role |
|---|---|
| `notebooks/front_athlete_preprocessing.ipynb` | Early exploration and feasibility prototype |
| `notebooks/player_angle_isolation_pipeline.ipynb` | Multi-player, multi-camera athlete isolation workflow |
| `notebooks/player_pose_estimation.ipynb` | Batch pose estimation and pose quality inspection |
| `notebooks/prepare_model_dataset.ipynb` | Label parsing, quality filtering, aggregation, and dataset export |
| `notebooks/build_score_model.ipynb` | Nested comparison of ML, neural, and attention architectures |
| `src/poomsae_scoring/preprocessing/isolation.py` | Reusable discovery, detection, tracking, masking, and cropping logic |
| `src/poomsae_scoring/scoring/pose_estimation.py` | Reusable MediaPipe video pipeline |
| `src/poomsae_scoring/scoring/pose_features.py` | Geometric, balance, and motion feature calculations |
| `src/poomsae_scoring/scoring/dataset_preparation.py` | Score parsing, view summarization, joins, validation, and exports |

<div dir="rtl">
<b>راهنمای سریع:</b> برای مشاهدهٔ مسیر آزمایشی اولیه به نوت‌بوک تشخیص ورزشکار جلویی مراجعه کنید؛ برای جداسازی کل داده از نوت‌بوک جداسازی زاویه‌ها؛ برای استخراج اسکلت از نوت‌بوک تخمین وضعیت؛ برای ساخت جدول نهایی از نوت‌بوک آماده‌سازی دیتاست؛ و برای مقایسه و ذخیرهٔ مدل‌ها از نوت‌بوک ساخت مدل استفاده کنید. منطق قابل استفادهٔ مجدد در پوشهٔ <code>src/poomsae_scoring</code> قرار دارد.
</div>

## 20. Final conclusion

### English

The project now has a complete experimental pipeline from multi-camera video to score prediction. The most important achievement is not a particular algorithm; it is the construction of a traceable, leakage-aware data path in which athlete identity, camera view, pose quality, judge labels, preprocessing, validation, and saved artifacts are explicitly controlled.

The current model comparison can identify the most promising baseline for this dataset, but it cannot remove the fundamental uncertainty caused by 54 narrow-range labels. The scientifically sound next step is to execute and audit the benchmark, then improve the amount and granularity of labeled data before increasing model complexity.

### فارسی

<div dir="rtl">
پروژه اکنون یک مسیر آزمایشی کامل از ویدئوی چنددوربینه تا پیش‌بینی امتیاز دارد. مهم‌ترین دستاورد، انتخاب یک الگوریتم خاص نیست؛ بلکه ساخت یک زنجیرهٔ دادهٔ قابل ردیابی و مقاوم در برابر نشت اطلاعات است که در آن هویت ورزشکار، زاویهٔ دوربین، کیفیت تخمین وضعیت، برچسب داور، پیش‌پردازش، اعتبارسنجی و خروجی‌های ذخیره‌شده به‌صورت صریح کنترل می‌شوند.

مقایسهٔ فعلی می‌تواند مناسب‌ترین مدل پایه را برای این دیتاست مشخص کند، اما عدم‌قطعیت ناشی از فقط ۵۴ برچسب با دامنهٔ محدود را از بین نمی‌برد. مسیر علمی صحیح این است که ابتدا آزمایش مدل‌ها کامل اجرا و ممیزی شود، سپس پیش از افزایش پیچیدگی معماری، تعداد و جزئیات داده‌های برچسب‌خورده افزایش یابد.
</div>